In [ ]:
import pypsa
import plotly.express as px
import pandas as pd
import calendar
import plotly.graph_objs as go

In [ ]:
n_gurobi = pypsa.Network("../results/2030_gurobi_conditioned-dispatch,storage_segmented-2920.nc")
n_pdlp = pypsa.Network("../results/2030_cupdlpx_conditioned-dispatch,storage_segmented-2920.nc")

In [ ]:
def energy_balance(network, grouper: str):
    df = network.statistics.energy_balance(bus_carrier="AC", groupby_time=False).dropna(how="all").droplevel(["component", "bus_carrier"]).T
    df_grouped = df.groupby(getattr(network.snapshots, grouper)).sum()

    return df_grouped

In [ ]:
nc = pypsa.NetworkCollection({"gurobi": n_gurobi, "pdlp": n_pdlp})

In [ ]:
eb = energy_balance(nc, "year")

df_plot = eb.unstack().to_frame("Energy MWh").reset_index()

In [ ]:
fig = px.bar(
    df_plot.sort_values(["Energy MWh", "carrier"]),
    x="carrier",
    y="Energy MWh",
    color="network",
    height=600,
    category_orders={"carrier": df_plot.sort_values(["Energy MWh", "carrier"]).carrier.drop_duplicates().values},
    title="Annual energy balance"
)
fig.update_layout(barmode="group")

In [ ]:
def get_carrier_mapping(network, component, carrier):
    idx = network.components[component].static.query(f"carrier in {carrier}").carrier.reset_index("network", drop=True)
    idx = idx[~idx.index.duplicated()]
    return idx

In [ ]:
idx = get_carrier_mapping(nc, "StorageUnit", ["PHS"])
df_plot = nc.storage_units_t.state_of_charge.resample("7D").sum().stack("network").T.groupby(idx).sum().unstack().to_frame("Energy MWh").reset_index()
fig = px.line(df_plot, x="snapshot", y="Energy MWh", color="network", height=500, title="Weekly PHS state of charge")
fig.update_traces(opacity=0.5)

In [ ]:
idx = get_carrier_mapping(nc, "Store", ["EV DSR"])
idx = idx.to_frame().assign(country=idx.index.str[:2])
df_plot = nc.stores_t.e.stack("network").T.groupby(idx.country).sum().unstack().to_frame("Energy MWh").reset_index()
px.line(df_plot, x="snapshot", y="Energy MWh", color="country", facet_col="network", height=500)

In [ ]:
idx = get_carrier_mapping(nc, "Generator", ["CCGT"])
df_plot = nc.generators_t.p.resample("1D").sum().stack("network").T.groupby(idx).sum().unstack().to_frame("Energy MWh").reset_index()
fig = px.line(df_plot, x="snapshot", y="Energy MWh", color="network", height=500, title="CCGT daily dispatch")
fig.update_traces(opacity=0.5)

In [ ]:
idx = get_carrier_mapping(nc, "Link", ["EV V2G"])
df_plot = nc.links_t.p0.resample("1D").sum().stack("network").T.groupby(idx).sum().unstack().to_frame("Energy MWh").reset_index()
fig = px.line(df_plot, x="snapshot", y="Energy MWh", color="network", height=500, title="EV V2G dispatch")
fig.update_traces(opacity=0.5)

In [ ]:
def plot_corr(component, carrier):
    idx = get_carrier_mapping(nc, component, [carrier])
    df_plot = (
        nc.components[component]
            .dynamic["p" if component != "Link" else "p0"]
            .stack("network")
            .T
            .groupby(idx).sum()
            .stack("snapshot")
            .droplevel("carrier")
            .mul(nc.snapshot_weightings["objective"], axis=0)
    )
    fig = px.scatter(df_plot, x="gurobi", y="pdlp", title=f"{component} - {carrier} - Timeseries correlation")
    fig.update_traces(opacity=0.4)
    fig.add_annotation(text=f"Pearson Correlation: {df_plot.corr().round(2).loc["gurobi", "pdlp"]}", x=0, y=1, xref="paper", yref="paper", showarrow=False)
    fig.add_trace(go.Scatter(x=[df_plot.min().min(), df_plot.max().max()], y=[df_plot.min().min(), df_plot.max().max()], showlegend=False, mode="lines"))
    return fig
plot_corr("Link", "EV V2G")

In [ ]:
plot_corr("Generator", "CCGT")

In [ ]:
plot_corr("StorageUnit", "PHS")

In [ ]:
plot_corr("Store", "H2 Store")

In [ ]:
df_plot = nc.statistics.opex().to_frame("opex").reset_index()
px.bar(df_plot, x="network", y="opex", color="carrier")